In [2]:
# ===== IMPORT LIBRARIES =====
import os
import torch
import tqdm
from sklearn.model_selection import train_test_split

from monai.data import DataLoader, CacheDataset
from monai.networks.nets import UNet
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.losses import DiceLoss, DiceCELoss
from monai.metrics import DiceMetric, ConfusionMatrixMetric
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    ScaleIntensityd, SpatialPadd, RandCropByPosNegLabeld,
    RandFlipd, RandRotate90d, RandAffined, RandBiasFieldd,
    RandGaussianNoised, RandAdjustContrastd,
    ToTensord, EnsureTyped, AsDiscreted
)
from monai.networks.utils import one_hot


In [3]:
# Set seed for reproducibility
set_determinism(seed=42)

# ======= CREATING DICTS =======
def get_data_list(root_dir):
    data_dicts = []
    for patient_id in os.listdir(root_dir):
        patient_path = os.path.join(root_dir, patient_id, "preRT")
        if not os.path.isdir(patient_path):
            print(f"⚠️ Le répertoire {patient_path} n'existe pas ou n'est pas un répertoire.")
            continue
        # Trouver fichiers T2 et masque
        t2_files = os.path.join(patient_path, f"{patient_id}_preRT_T2.nii.gz")
        mask_files = os.path.join(patient_path, f"{patient_id}_preRT_mask.nii.gz")

        if t2_files and mask_files:
            data_dicts.append({
                "image": t2_files,
                "label": mask_files
            })
        else:
            print(f"⚠️ Fichiers manquants pour le patient {patient_id}")
    return data_dicts

# Construction des jeux d'entraînement et de validation
train_data_dir = "/cluster/projects/vc/data/mic/open/HNTS-MRG/train"
test_data_dir = "/cluster/projects/vc/data/mic/open/HNTS-MRG/test"

print("Searching datas...")
# Obtenir les données d'entraînement complètes
train_data_full = get_data_list(train_data_dir)

# Diviser les données d'entraînement en train et validation
train_data, val_data = train_test_split(train_data_full, train_size=0.8, test_size=0.2, random_state=42)

# Obtenir les données de test
test_data = get_data_list(test_data_dir)

# Exemple d'aperçu
print(f"Train data: {len(train_data)} samples")
print(f"Validation data: {len(val_data)} samples")
print(f"Test data: {len(test_data)} samples")

# Exemple d'aperçu
print(train_data[0])
# {'image': './train/10/preRT/10_preRT_T2.nii.gz', 'label': './train/10/preRT/10_preRT_mask.nii.gz'}

Searching datas...
Train data: 104 samples
Validation data: 26 samples
Test data: 20 samples
{'image': '/cluster/projects/vc/data/mic/open/HNTS-MRG/train/23/preRT/23_preRT_T2.nii.gz', 'label': '/cluster/projects/vc/data/mic/open/HNTS-MRG/train/23/preRT/23_preRT_mask.nii.gz'}


In [ ]:
# ======= TRANSFORMING TRAINING DATAS =======

print("Transforming datas...")
# Transforms
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityd(keys=["image"]),
    SpatialPadd(keys=["image", "label"], spatial_size=(96, 96, 96)),

    # Crop positive and negative patches
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label",
        spatial_size=(96, 96, 96), pos=1, neg=1, num_samples=4,
        image_key="image", image_threshold=0
    ),

    # Spatial augmentations
    RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandAffined(
        keys=["image", "label"],
        prob=0.5,
        rotate_range=(0.1, 0.1, 0.1),
        shear_range=None,
        translate_range=(10, 10, 5),
        scale_range=(0.1, 0.1, 0.1),
        mode=("bilinear", "nearest")
    ),

    # Intensity augmentations
    RandBiasFieldd(keys=["image"], prob=0.3),
    RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.1),
    RandAdjustContrastd(keys=["image"], prob=0.3, gamma=(0.7, 1.5)),

    ToTensord(keys=["image", "label"]),
])
val_transforms = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityd(keys=["image"]),
    ToTensord(keys=["image", "label"]),
]

# import nibabel as nib
# for entry in train_data:
#     if not os.path.exists(entry["image"]):
#         print(f"❌ Image not found: {entry['image']}")
#     if not os.path.exists(entry["label"]):
#         print(f"❌ Label not found: {entry['label']}")
#     img = nib.load(entry["image"])
#     print(img.shape)
#     label = nib.load(entry["label"])
#     print(label.shape)

# Datasets and Loaders
train_ds = CacheDataset(data=train_data, transform=train_transforms, cache_rate=1.0)
train_loader = DataLoader(train_ds, batch_size=3, shuffle=True)
    # print(cm_stats)

val_ds = CacheDataset(data=val_data, transform=val_transforms, cache_rate=1.0)
val_loader = DataLoader(val_ds, batch_size=1)


Transforming datas...


Loading dataset: 100%|██████████| 26/26 [00:45<00:00,  1.77s/it]


In [38]:
for val_data in val_loader:
    val_images, val_labels = val_data["image"].cuda(), val_data["label"].cuda()
    print(f"Validation images shape: {val_images.shape}")
    print(f"Validation labels shape: {val_labels.shape}")
    break
for train_data in train_loader:
    train_images, train_labels = train_data["image"].cuda(), train_data["label"].cuda()
    print(f"Train images shape: {train_images.shape}")
    print(f"Train labels shape: {train_labels.shape}")
    break
# Validation images shape: torch.Size([1, 1, 171, 171, 77])
# Validation labels shape: torch.Size([1, 1, 171, 171, 77])
# Train images shape: torch.Size([12, 1, 96, 96, 96])
# Train labels shape: torch.Size([12, 1, 96, 96, 96])

Validation images shape: torch.Size([1, 1, 171, 171, 77])
Validation labels shape: torch.Size([1, 1, 171, 171, 77])
Train images shape: torch.Size([12, 1, 96, 96, 96])
Train labels shape: torch.Size([12, 1, 96, 96, 96])


In [ ]:
# ====== MODEL ======

# Check if CUDA is available and set the device accordingly
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
    device = torch.device("cuda")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")

# Define the model
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=6,
    dropout=0.3,
).to(device)

# ====== LOSS ======
# classic one
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
# loss_function = DiceLoss()

# Dice + CrossEntropy Loss
# This is a good choice if you have a multi-class segmentation problem and want to balance the contribution of each class.
# loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False)
# loss_function = DiceCELoss(include_background=False)

# Adjust alpha (false negative penalty) and beta (false positive penalty) based on your task. This is particularly good if your tumor is very small in volume.
# loss_function = TverskyLoss(to_onehot_y=True, softmax=True, alpha=0.7, beta=0.3)

# ====== METRICS ======
dice_metric = DiceMetric(include_background=False, reduction="mean")
confusion_metric = ConfusionMatrixMetric(include_background=False, reduction="mean", metric_name="all")

# ====== OPTIMIZER ======
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)
max_epochs = 5

CUDA is available. Using GPU.


/cluster/home/vutourni/HNTS-MRG/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


In [51]:
# ====== TRAINING ======
print("Training...")
# Training loop
best_dice = 0.0
for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0

    tqdm_bar = tqdm.tqdm(train_loader, desc="Training", unit="batch")
    for batch_data in tqdm_bar:
        tqdm_bar.set_description(f"Epoch {epoch+1}/{max_epochs} - Training")
        tqdm_bar.set_postfix(loss=epoch_loss)
        tqdm_bar.refresh()

        
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        # print(f"Batch size: {inputs.shape[0]}")
        # print(f"Inputs shape: {inputs.shape}")
        # print(f"Outputs shape: {outputs.shape}")
        
        # labels = one_hot(labels, num_classes=outputs.shape[1])
        # print(f"Labels shape: {labels.shape}")

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        tqdm_bar.set_postfix(loss=epoch_loss / len(train_loader))
        tqdm_bar.refresh()

    print(f"Train loss: {epoch_loss / len(train_loader):.4f}")

    # # Validation
    # model.eval()
    with torch.no_grad():
        dice_metric.reset()
        # confusion_metric.reset()
        tqdm_bar = tqdm.tqdm(val_loader, desc="Validation", unit="batch")
        for val_data in tqdm_bar:
            tqdm_bar.set_description(f"Epoch {epoch+1}/{max_epochs} - Validation")
            tqdm_bar.refresh()

            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)

            val_outputs = sliding_window_inference(val_inputs, (96, 96, 96), sw_batch_size=4, predictor=model)
            
            # print(f"Inputs shape: {val_inputs.shape}")
            # print(f"Outputs shape: {val_outputs.shape}")
            # print(f"Labels shape: {val_labels.shape}")

            # val_labels = one_hot(val_labels, num_classes=val_outputs.shape[1])
            # print(f"Labels shape: {labels.shape}")

            dice_metric(y_pred=val_outputs, y=val_labels)
            # confusion_metric(y_pred=val_outputs, y=val_labels)
            tqdm_bar.set_postfix(dice=dice_metric.aggregate().item())
            tqdm_bar.refresh()

        dice_score = dice_metric.aggregate().item()
        print(f"Validation Dice: {dice_score:.4f}")

        dice_metric.reset()

        # save best model
        if epoch == 0 or dice_score > best_dice:
            best_dice = dice_score
            torch.save(model.state_dict(), os.path.join("../results","best_model.pth"))
            print(f"Best model saved with Dice: {best_dice:.4f}")
        else:
            print(f"Model not improved. Current best Dice: {best_dice:.4f}")

# Save the final model
torch.save(model.state_dict(), os.path.join("../results","final_model.pth"))
print("Training completed.")



Training...
Epoch 1/5


Epoch 1/5 - Training: 100%|██████████| 35/35 [03:35<00:00,  6.16s/batch, loss=1.11] 


Train loss: 1.1113


Epoch 1/5 - Validation: 100%|██████████| 26/26 [00:49<00:00,  1.89s/batch, dice=0.00187]


Validation Dice: 0.0019
Best model saved with Dice: 0.0019
Epoch 2/5


...

Train loss: 1.0451


Epoch 2/5 - Validation: 100%|██████████| 26/26 [00:49<00:00,  1.88s/batch, dice=0.00187]


Validation Dice: 0.0019
Model not improved. Current best Dice: 0.0019
Epoch 3/5


Epoch 3/5 - Training:   9%|▊         | 3/35 [00:26<04:42,  8.84s/batch, loss=3.07]  


KeyboardInterrupt: 

In [4]:
# ======= LOADING TEST DATAS =======

test_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityd(keys=["image"]),
    ToTensord(keys=["image", "label"]),
    # AsDiscreted(keys="label")
])

test_ds = CacheDataset(data=test_data, transform=test_transforms, cache_rate=1.0)
test_loader = DataLoader(test_ds, batch_size=1)

Loading dataset:   0%|          | 0/20 [00:00<?, ?it/s]

Loading dataset: 100%|██████████| 20/20 [00:56<00:00,  2.84s/it]


In [5]:
for test_data in test_loader:
    test_inputs = test_data["image"]
    test_labels = test_data["label"]
    print(test_inputs.shape)
    print(test_labels.shape)
    break
# torch.Size([1, 1, 171, 171, 77])
# torch.Size([1, 1, 171, 171, 77])

torch.Size([1, 1, 171, 171, 77])
torch.Size([1, 1, 171, 171, 77])


In [13]:
# ====== LOADING MODEL ======
# model_name = "best_model.pth"
model_name = "model_epoch_15.pth"
model_path = os.path.join("../results", model_name)
model.load_state_dict(torch.load(model_path))

<All keys matched successfully>

In [14]:
# ====== METRICS ======
test_dice_metric = DiceMetric(include_background=False, reduction="mean")
metrics_list = [ "sensitivity", "specificity", "accuracy", "precision", "f1"]
confusion_metric = ConfusionMatrixMetric(include_background=False, reduction="mean", metric_name=metrics_list)

# ===== TESTING =====
print("Testing...")
model.eval()

with torch.no_grad():
    test_dice_metric.reset()
    confusion_metric.reset()

    tqdm_bar = tqdm.tqdm(test_loader, desc="Testing", unit="batch")
    for test_data in tqdm_bar:
        tqdm_bar.set_description("Testing")
        tqdm_bar.refresh()

        test_inputs = test_data["image"].to(device)
        test_labels = test_data["label"].to(device)
        # print(f"Test inputs shape: {test_inputs.shape}")     
        
        test_outputs = sliding_window_inference(test_inputs, (96, 96, 96), sw_batch_size=4, predictor=model)

        # print(f"Test outputs shape: {test_outputs.shape}")                
        # test_outputs = torch.argmax(test_outputs, dim=1, keepdim=True)
        # print(f"Test outputs shape: {test_outputs.shape}")

        # print(f"Test labels shape: {test_labels.shape}")   
        test_labels = one_hot(test_labels, num_classes=test_outputs.shape[1])
        # print(f"Test labels shape: {test_labels.shape}")   

        test_dice_metric(y_pred=test_outputs, y=test_labels)
        confusion_metric(y_pred=test_outputs, y=test_labels)
        tqdm_bar.set_postfix(dice=test_dice_metric.aggregate().item())
        tqdm_bar.refresh()

    test_metric = test_dice_metric.aggregate().item()
    test_dice_metric.reset()

    cm_stats = confusion_metric.aggregate()

    # print(cm_stats)
    for name, value in zip(metrics_list, cm_stats):
        print(f"{name}: {value.item():.4f}")
    # f1, precision, accuracy, sensitivity, specificity = cm_stats
    # print(f"F1: {f1.item():.4f}")
    # print(f"Precision: {precision.item():.4f}")
    # print(f"Accuracy: {accuracy.item():.4f}")
    # print(f"Sensitivity: {sensitivity.item():.4f}")
    # print(f"Specificity: {specificity.item():.4f}")
    # tp, fp, tn, fn = cm_stats["tp"], cm_stats["fp"], cm_stats["tn"], cm_stats["fn"]

    # print("Confusion Matrix stats:")
    # print("True Positives:", tp)
    # print("False Positives:", fp)
    # print("True Negatives:", tn)
    # print("False Negatives:", fn)
    confusion_metric.reset()

    print(f"Test Dice: {test_metric:.4f}")


Testing...


Testing: 100%|██████████| 20/20 [00:58<00:00,  2.94s/batch, dice=0.00288]

sensitivity: 0.0000
specificity: 0.0000
accuracy: 0.0000
precision: 0.0000
f1: 0.0000
Test Dice: 0.0029
